In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding

In [4]:
input_texts = []
target_texts = []

from google.colab import files

uploaded = files.upload()  # Select ara.txt
data_path = "/content/ara.txt"

with open(data_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for line in lines[:30000]:
    parts= line.split('\t')

    if len(parts) >= 2:
        input_text=parts[0].strip()
        target_text=parts[1].strip()

        target_text= "<start> " + target_text + " <end>"

        input_texts.append(input_text)
        target_texts.append(target_text)

print(input_texts[:5])
print(target_texts[:5])

Saving ara.txt to ara.txt
['Hi.', 'Run!', 'Help!', 'Jump!', 'Stop!']
['<start> مرحبًا. <end>', '<start> اركض! <end>', '<start> النجدة! <end>', '<start> اقفز! <end>', '<start> قف! <end>']


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

input_tokenizer = AutoTokenizer.from_pretrained("PontifexMaximus/ArabicTranslator")
target_tokenizer = AutoTokenizer.from_pretrained("PontifexMaximus/ArabicTranslator")

input_batch = input_tokenizer(input_texts, return_tensors='pt', padding=True,truncation=True)
target_batch = target_tokenizer(target_texts, return_tensors='pt', padding=True,truncation=True)

input_sequences = input_batch["input_ids"]
target_sequences = target_batch["input_ids"]

num_encoder_tokens = len(input_tokenizer) +1
num_decoder_tokens = len(target_tokenizer) +1

print('Input Vocab length:', num_encoder_tokens)
print('Target Vocab length:', num_decoder_tokens)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

source.spm: reconstructing file:   0%|          |  0.00B /  917kB            

source.spm: downloading bytes:           |  0.00B            

target.spm: reconstructing file:   0%|          |  0.00B /  802kB            

target.spm: downloading bytes:           |  0.00B            

vocab.json:   0%|          | 0.00/2.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Input Vocab length: 62835
Target Vocab length: 62835


In [6]:
max_encoder_seq_length = max(len(seq) for seq in input_sequences)
max_decoder_seq_length = max(len(seq) for seq in target_sequences)

encoder_input_data = pad_sequences(input_sequences, maxlen=max_encoder_seq_length, padding='post')
decoder_input_data = pad_sequences(target_sequences, maxlen=max_decoder_seq_length, padding='post')

print('Max input seq length:', max_encoder_seq_length)
print('Max target seq length:', max_decoder_seq_length)

Max input seq length: 76
Max target seq length: 69


In [7]:
decoder_target_data = np.zeros_like(decoder_input_data)
decoder_target_data[:, :-1] = decoder_input_data[:, 1:]
decoder_target_data[:, -1] = 0

In [8]:
embedding_dim = 100

encoder_inputs = Input(shape=(max_encoder_seq_length,))
encoder_embedding = Embedding(num_encoder_tokens, embedding_dim,mask_zero=True)(encoder_inputs)
embedding_lstm = LSTM(256, return_state=True, dropout=0.3, recurrent_dropout=0.3)
encoder_outputs, state_h, state_c = embedding_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(max_decoder_seq_length,))
decoder_embedding = Embedding(num_decoder_tokens, embedding_dim,mask_zero=True)(decoder_inputs)
decoder_lstm = LSTM(256, return_sequences=True, return_state=True, dropout=0.3, recurrent_dropout=0.3)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
print("Model Compiled Successfully")

Model Compiled Successfully


In [9]:
print(encoder_input_data.shape)
print(decoder_input_data.shape)
print(decoder_target_data.shape)

(10842, 76)
(10842, 69)
(10842, 69)


In [10]:
#model = AutoModelForSeq2SeqLM.from_pretrained("PontifexMaximus/ArabicTranslator", device_map="auto")
#model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics= ['accuracy'])

In [11]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

base_model = "Helsinki-NLP/opus-mt-en-ar"
tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForSeq2SeqLM.from_pretrained(base_model)

# input_texts and target_texts are from ara.txt.
# Strip the Keras-only <start> and <end> markers.
examples = Dataset.from_dict({
    "translation": [
        {"en": en, "ar": ar.replace("<start> ", "").replace(" <end>", "")}
        for en, ar in zip(input_texts, target_texts)
    ]
})

def tokenize_batch(batch):
    sources = [item["en"] for item in batch["translation"]]
    targets = [item["ar"] for item in batch["translation"]]
    result = tokenizer(sources, text_target=targets, max_length=128, truncation=True)
    return result

tokenized = examples.map(tokenize_batch, batched=True, remove_columns=["translation"])

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/801k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/917k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.12M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Map:   0%|          | 0/10842 [00:00<?, ? examples/s]

In [12]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset

# Split into train and validation sets
split = tokenized.train_test_split(test_size=0.1, seed=42)

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/arabic-translator",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    logging_steps=50,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
)

trainer.train()
trainer.save_model("/content/arabic-translator")
tokenizer.save_pretrained("/content/arabic-translator")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss
1,1.087614,1.014455
2,0.780620,1.005900
3,0.681327,1.008827


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/arabic-translator/tokenizer_config.json',
 '/content/arabic-translator/vocab.json',
 '/content/arabic-translator/source.spm',
 '/content/arabic-translator/target.spm',
 '/content/arabic-translator/added_tokens.json')

In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_path = "/content/arabic-translator"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

def translate(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [23]:
print(translate("My name is Mohammed Nasser"))
print(translate("I'm 27 years old"))

اسمي محمد ناصر
عمري 27 عاماً
